# 41. LS 코퍼스 빈도 검색

30번 사전검색 결과(word 리스트)의 실제 사용 빈도를 LS 코퍼스에서 조회

## 입력
- 30번 결과 CSV: `search_results/*.csv`
- LS 빈도 CSV: `00_raw_data/02_nikl_ls/07_ALL_word_freq.csv` (102MB)

## 출력
- 빈도 보강 CSV: `search_results/ls_freq_*.csv`

## 1. 환경 설정

In [1]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.getcwd())
    print(f'Local mode: {PROJECT_ROOT}')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# 경로 설정
SEARCH_RESULTS_30 = f'{PROJECT_ROOT}/30_search_dictionary/search_results'
SEARCH_RESULTS_37 = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
LS_FREQ_PATH = f'{PROJECT_ROOT}/00_raw_data/02_nikl_ls/07_ALL_word_freq.csv'
RESULT_DIR = f'{PROJECT_ROOT}/40_search_LS_corpus/search_results'
os.makedirs(RESULT_DIR, exist_ok=True)

print(f'30번 결과: {SEARCH_RESULTS_30}')
print(f'37번 결과: {SEARCH_RESULTS_37}')
print(f'LS 빈도: {LS_FREQ_PATH}')

30번 결과: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results
37번 결과: /content/drive/MyDrive/DATA_2026/37_vowel_harmony_collision/search_results
LS 빈도: /content/drive/MyDrive/DATA_2026/00_raw_data/02_nikl_ls/07_ALL_word_freq.csv


## 2. 30번 검색 결과 로드

In [3]:
# 30번 + 37번 결과 CSV 목록
result_files_30 = list(Path(SEARCH_RESULTS_30).glob('*.csv'))
result_files_37 = [f for f in Path(SEARCH_RESULTS_37).glob('*.csv')
                   if not f.name.startswith(('ls_freq_', 'seoul_', 'dialogue_'))]
result_files = result_files_30 + result_files_37

for f in result_files:
    print(f'  {f.parent.parent.name}/{f.name} ({f.stat().st_size / 1024 / 1024:.1f}MB)')

# 각 현상별 word 리스트 추출
phenomena = {}
for f in result_files:
    name = f.stem
    df_tmp = pd.read_csv(f, encoding='utf-8-sig', usecols=['word'])
    words = df_tmp['word'].unique().tolist()
    phenomena[name] = words
    print(f'  {name}: {len(words):,}개 단어')

# 전체 고유 단어 리스트
all_words = set()
for ws in phenomena.values():
    all_words.update(ws)
print(f'\n전체 고유 단어: {len(all_words):,}개')

  30_search_dictionary/n_l_insertion_candidates_v2_20260210_130456.csv (2.5MB)
  30_search_dictionary/nl_ln_nasalization_candidates_20260210_130654.csv (4.9MB)
  30_search_dictionary/n_l_insertion_candidates_v2_20260312_064714.csv (12.3MB)
  30_search_dictionary/n_l_insertion_candidates_v2_20260312_070223.csv (12.2MB)
  30_search_dictionary/nl_ln_nasalization_candidates_20260312_070559.csv (8.6MB)
  30_search_dictionary/fortis_compound_candidates_20260312_071048.csv (45.1MB)
  30_search_dictionary/fortis_compound_candidates_20260312_071626.csv (41.6MB)
  37_vowel_harmony_collision/vowel_harmony_collision_all_20260312_071836.csv (22.9MB)
  37_vowel_harmony_collision/vowel_collision_candidates_20260312_071836.csv (21.6MB)
  37_vowel_harmony_collision/vowel_harmony_mismatch_20260312_071836.csv (0.0MB)
  n_l_insertion_candidates_v2_20260210_130456: 6,149개 단어
  nl_ln_nasalization_candidates_20260210_130654: 12,075개 단어
  n_l_insertion_candidates_v2_20260312_064714: 21,818개 단어
  n_l_insertion

## 3. LS 빈도 데이터 로드

In [4]:
# LS 전체 빈도 로드 (102MB)
df_ls = pd.read_csv(LS_FREQ_PATH, encoding='utf-8-sig', low_memory=False)
print(f'LS 빈도: {len(df_ls):,}행')
print(f'컬럼: {list(df_ls.columns)}')
print(f'\ncorpus_type 분포:')
print(df_ls['corpus_type'].value_counts())
df_ls.head(3)

LS 빈도: 817,688행
컬럼: ['word_surface', 'morpheme_analysis', 'sense_analysis', 'corpus_type', 'corpus_name', 'freq', 'word_roman', 'word_roman_mfa']

corpus_type 분포:
corpus_type
NXLS    498684
SXLS    183641
MXLS    135363
Name: count, dtype: int64


,word_surface,morpheme_analysis,sense_analysis,corpus_type,corpus_name,freq,word_roman,word_roman_mfa
0,수,수/NNB,수/3,NXLS,문어 말뭉치,9965,s0 uu,S U
1,등,등/NNB,등/10,NXLS,문어 말뭉치,8114,t0 xx ng,D EU NG
2,있다.,있/VX,있/23,NXLS,문어 말뭉치,7459,ii ss - t0 aa - .,I t - D A - .


## 4. 빈도 매칭

In [5]:
def lookup_ls_frequency(word_list, df_ls):
    """
    30번 word 리스트를 LS 빈도에서 조회

    Returns: DataFrame with word, corpus_type, corpus_name, freq
    """
    # word_surface 기준 매칭
    df_matched = df_ls[df_ls['word_surface'].isin(word_list)].copy()

    # 코퍼스 유형별 빈도 집계
    freq_by_type = df_matched.groupby(['word_surface', 'corpus_type'])['freq'].sum().reset_index()
    freq_pivot = freq_by_type.pivot_table(
        index='word_surface', columns='corpus_type', values='freq', fill_value=0
    ).reset_index()
    freq_pivot.columns.name = None

    # 총 빈도
    freq_cols = [c for c in freq_pivot.columns if c != 'word_surface']
    freq_pivot['freq_LS_total_raw'] = freq_pivot[freq_cols].sum(axis=1)

    # 매칭 실패 단어
    matched_words = set(df_matched['word_surface'].unique())
    unmatched = [w for w in word_list if w not in matched_words]

    print(f'매칭 성공: {len(matched_words):,} / {len(word_list):,}')
    print(f'매칭 실패: {len(unmatched):,}')
    if unmatched[:10]:
        print(f'  예시: {unmatched[:10]}')

    return freq_pivot, unmatched

# 전체 단어 빈도 조회
freq_result, unmatched = lookup_ls_frequency(list(all_words), df_ls)

매칭 성공: 11,281 / 204,656
매칭 실패: 193,375
  예시: ['방외자', '협상설', '육중대나마', '만고불멸하다', '유차하다', '보손장', '우김질', '여인당', '운학기', '냉각수']


In [6]:
# 결과 미리보기
print(f'\n빈도 매칭 결과: {len(freq_result):,}행')
print(freq_result.sort_values('freq_LS_total_raw', ascending=False).head(20))


빈도 매칭 결과: 11,281행
     word_surface   MXLS    NXLS   SXLS  freq_LS_total_raw
7527           이번  128.0  1949.0  265.0             2342.0
7052           요즘  950.0   234.0  528.0             1712.0
1234           국내   15.0  1355.0   45.0             1415.0
1034           관련   11.0  1260.0   75.0             1346.0
2855           등이    1.0  1109.0   21.0             1131.0
2482          대통령    1.0   835.0  238.0             1074.0
9050          지난달    0.0  1055.0    6.0             1061.0
7758           일이  149.0   407.0  366.0              922.0
9552          청와대    2.0   638.0   64.0              704.0
3638          민주당    0.0   662.0   27.0              689.0
1430          그동안   13.0   557.0   94.0              664.0
5081         새누리당    0.0   553.0   94.0              647.0
3070           말이   48.0   316.0  280.0              644.0
7951           작년   14.0   485.0   71.0              570.0
8291           전국    1.0   535.0   28.0              564.0
8985           중인    1.0   489.0   15

## 5. v7 빈도와 비교

In [7]:
# TODO: v7의 freq_LS_total과 LS 원본의 freq 차이 확인
# v7은 sense_no 단위, LS는 word_surface 단위로 집계 방식이 다를 수 있음
print('TODO: v7 freq_LS_total vs LS 원본 freq 비교')

TODO: v7 freq_LS_total vs LS 원본 freq 비교


## 6. 현상별 빈도 보강 CSV 저장

In [8]:
# 각 30번 결과 CSV에 LS 빈도 병합
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

for f in result_files:
    df_orig = pd.read_csv(f, encoding='utf-8-sig')

    # word 기준 빈도 조인
    df_merged = df_orig.merge(
        freq_result,
        left_on='word', right_on='word_surface',
        how='left', suffixes=('', '_ls_raw')
    )

    # 저장
    out_name = f'ls_freq_{f.stem}_{timestamp}.csv'
    out_path = f'{RESULT_DIR}/{out_name}'
    df_merged.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'  {out_name}: {len(df_merged):,}행')

print(f'\n저장 완료: {RESULT_DIR}')

  ls_freq_n_l_insertion_candidates_v2_20260210_130456_20260313_050953.csv: 6,628행
  ls_freq_nl_ln_nasalization_candidates_20260210_130654_20260313_050953.csv: 18,878행
  ls_freq_n_l_insertion_candidates_v2_20260312_064714_20260313_050953.csv: 32,761행
  ls_freq_n_l_insertion_candidates_v2_20260312_070223_20260313_050953.csv: 32,761행
  ls_freq_nl_ln_nasalization_candidates_20260312_070559_20260313_050953.csv: 29,218행
  ls_freq_fortis_compound_candidates_20260312_071048_20260313_050953.csv: 128,164행
  ls_freq_fortis_compound_candidates_20260312_071626_20260313_050953.csv: 128,164행
  ls_freq_vowel_harmony_collision_all_20260312_071836_20260313_050953.csv: 93,413행
  ls_freq_vowel_collision_candidates_20260312_071836_20260313_050953.csv: 87,271행
  ls_freq_vowel_harmony_mismatch_20260312_071836_20260313_050953.csv: 35행

저장 완료: /content/drive/MyDrive/DATA_2026/40_search_LS_corpus/search_results


## 7. 통계 요약

In [9]:
# 빈도 분포 통계
if len(freq_result) > 0:
    print('=== LS 빈도 분포 ===')
    print(freq_result['freq_LS_total_raw'].describe())
    print(f'\n빈도 0인 단어: {(freq_result["freq_LS_total_raw"] == 0).sum():,}')
    print(f'빈도 100+ 단어: {(freq_result["freq_LS_total_raw"] >= 100).sum():,}')
    print(f'빈도 1000+ 단어: {(freq_result["freq_LS_total_raw"] >= 1000).sum():,}')

=== LS 빈도 분포 ===
count    11281.000000
mean         8.741424
std         46.794525
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max       2342.000000
Name: freq_LS_total_raw, dtype: float64

빈도 0인 단어: 0
빈도 100+ 단어: 132
빈도 1000+ 단어: 7
